# DeferralX Colab Notebook

This notebook runs a faster, resume-safe DeferralX pipeline on Google Colab.

Run code cells in order. If collection is interrupted, rerun the collection cell with `--resume`.


In [1]:
# 1) Clone repo
!git clone https://github.com/elom354/deferralX
%cd deferralX


Cloning into 'deferralX'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 61 (delta 19), reused 61 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 35.74 KiB | 11.91 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/deferralX


In [1]:
# 2) Install dependencies
%cd deferralX
!python -m pip install --upgrade pip
!pip install "numpy<2" torch transformers datasets sentencepiece accelerate


/content/deferralX


In [2]:
# 2b) Runtime stability settings
import os
os.environ['KMP_USE_SHM'] = '0'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
print('Runtime env configured')


Runtime env configured


In [3]:
# 3) Build real questions from MMLU with domain/profile heterogeneity
!PYTHONPATH=src python -m deferralx.prepare_questions \
  --dataset cais/mmlu \
  --subset all \
  --split test \
  --output data/mmlu_questions_600.csv \
  --question-col question \
  --choices-col choices \
  --answer-col answer \
  --answer-is-index \
  --domain general \
  --domain-mode mmlu_subject \
  --profile-mode cycle \
  --severe-mode by_domain \
  --limit 600

README.md: 53.2kB [00:00, 105MB/s]
dataset_infos.json: 138kB [00:00, 183MB/s]
all/test-00000-of-00001.parquet: 100% 3.50M/3.50M [00:00<00:00, 5.59MB/s]
all/validation-00000-of-00001.parquet: 100% 408k/408k [00:00<00:00, 1.94MB/s]
all/dev-00000-of-00001.parquet: 100% 76.5k/76.5k [00:00<00:00, 365kB/s]
all/auxiliary_train-00000-of-00001.parqu(…): 100% 47.5M/47.5M [00:02<00:00, 23.6MB/s]
Generating test split: 100% 14042/14042 [00:00<00:00, 161804.68 examples/s]
Generating validation split: 100% 1531/1531 [00:00<00:00, 198979.90 examples/s]
Generating dev split: 100% 285/285 [00:00<00:00, 89240.51 examples/s]
Generating auxiliary_train split: 100% 99842/99842 [00:00<00:00, 144739.30 examples/s]
Wrote 600 rows to data/mmlu_questions_600.csv


In [4]:
# 4) Collect local HF logs (faster + resume-safe)
# Re-run this cell if Colab disconnects; --resume continues from last processed row.
!PYTHONPATH=src python -m deferralx.run collect-local-hf \
  --questions data/mmlu_questions_600.csv \
  --output data/real_llm_logs_local.csv \
  --audit-jsonl outputs/audit/real_collection_local_hf.jsonl \
  --model-id Qwen/Qwen2.5-0.5B-Instruct \
  --device auto \
  --max-tokens 96 \
  --agreement-samples 1 \
  --skip-confidence-pass \
  --resume


Resume mode: 0 already processed, 600 remaining.
config.json: 100% 659/659 [00:00<00:00, 3.05MB/s]
tokenizer_config.json: 7.30kB [00:00, 19.6MB/s]
vocab.json: 2.78MB [00:00, 67.4MB/s]
merges.txt: 1.67MB [00:00, 7.85MB/s]
tokenizer.json: 7.03MB [00:00, 95.2MB/s]
model.safetensors: 100% 988M/988M [00:06<00:00, 149MB/s]
Loading weights: 100% 290/290 [00:00<00:00, 382.16it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 242/242 [00:00<00:00, 1.64MB/s]
[1/600] general_000000 done | correct=0 | agreement=0.000
[2/600] general_000001 done | correct=0 | agreement=0.000
[3/600] general_000002 done | correct=0 | agreement=0.000
[4/600] general_000003 done | correct=0 | agreement=0.000
[5/600] general_000004 done | correct=0 | agreement=0.000
[6/600] general_000005 done | correct=0 | agreement=0.000
[7/600] general_000006 done | correct=0 | agreement=0.000
[8/600] general_000007 done | correct=0 | agreement=0.000
[9/600] general_000008 done | correct=0 | agreement=0.000
[10

In [5]:
# 5) Evaluate routing policies
!PYTHONPATH=src python -m deferralx.run run \
  --input data/real_llm_logs_local.csv \
  --outdir outputs/main_colab \
  --utility-config configs/utility_config.json \
  --test-ratio 0.30 \
  --seed 42 \
  --bootstrap 200


Experiment completed.
Results written to: outputs/main_colab


In [6]:
# 6) View results
!cat outputs/main_colab/summary.md
!head -n 20 outputs/main_colab/metrics_overall.csv


# DeferralX Experiment Summary

- Total samples: 600
- Train samples: 425
- Test samples: 175

## Best Overall Policy

- always_escalate: utility=-0.4011, coverage=0.0000, accepted_accuracy=NA, severe_error_rate=0.0000

## Best Policy by Domain

- finance: always_escalate (utility=-0.4071)
- general: always_escalate (utility=-0.4000)
- medical: always_escalate (utility=-0.4000)

## Best Policy by User Profile

- balanced_user: always_escalate (utility=-0.4000)
- cautious_novice: always_escalate (utility=-0.6000)
- expert_fast: always_escalate (utility=-0.2000)

## Heterogeneity Check

- Overall best policy: always_escalate
- Distinct best policies across domains: ['always_escalate']
- Distinct best policies across profiles: ['always_escalate']
policy,slice_name,n,utility_mean,coverage,escalation_rate,accepted_accuracy,severe_error_rate
always_escalate,overall,175,-0.401143,0.000000,1.000000,,0.000000
global_threshold,overall,175,-0.401143,0.000000,1.000000,,0.000000
domain_threshold,ov

In [7]:
# 7) Optional: download artifacts
from google.colab import files
files.download('outputs/main_colab/summary.md')
files.download('outputs/main_colab/metrics_overall.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>